In [ ]:
from google.colab import drive
drive.mount('/content/drive')

Mounted at /content/drive


In [ ]:
import os
import sys


PROJECT_ROOT = '/content/drive/MyDrive/Plagiarism-Detection-System'

os.chdir(PROJECT_ROOT)
print(f"Current Working Directory: {os.getcwd()}")
sys.path.append(PROJECT_ROOT)

Current Working Directory: /content/drive/MyDrive/Plagiarism-Detection-System


In [ ]:
!pip install -r requirements.txt

  Cloning https://github.com/helemanc/whisper.git to /tmp/pip-req-build-d_zfbt8s
  Running command git clone --filter=blob:none --quiet https://github.com/helemanc/whisper.git /tmp/pip-req-build-d_zfbt8s
  Resolved https://github.com/helemanc/whisper.git to commit a4796f6b3a241f8b63a8eb7df4ee1987ec62c475
  Installing build dependencies ... done
  Getting requirements to build wheel ... done
  Preparing metadata (pyproject.toml) ... done
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 43.6/43.6 kB 3.5 MB/s eta 0:00:00
  Created wheel for openai-whisper: filename=openai_whisper-20240930-py3-none-any.whl size=805487 sha256=4e846624be3c0b7b485179e4ea3ae8c6c915b880ea1e4dfdcc53dc663e70aef4
  Stored in directory: /tmp/pip-ephem-wheel-cache-etuqjlxg/wheels/31/45/41/0aface78dad26e2b915a8c1edf9598f22e0f8a9dbb80b01e74
Successfully built openai-whisper


In [ ]:
import os

%cd /content/drive/MyDrive/Plagiarism-Detection-System/

!mkdir -p models/clews
!mkdir -p models/wealy

if not os.path.exists("models/wealy/checkpoint.pt"):
    print("Downloading WEALY (from Hugging Face)...")
    !wget -O models/wealy/checkpoint.pt https://huggingface.co/audio-based-lyrics-matching/wealy-whisper-dvi/resolve/main/checkpoint_best.ckpt
else:
    print("WEALY model already exists. Skipping download.")

if not os.path.exists("models/clews/checkpoint.pt"):
    print("\nDownloading CLEWS (from Zenodo)...")
    !wget -O clews_checkpoints.zip "https://zenodo.org/records/15045900/files/clews.zip?download=1"
    !unzip clews_checkpoints.zip -d downloaded_checkpoints

    !find downloaded_checkpoints -name "*best.ckpt" | grep "dvi" | head -n 1 | xargs -I {} cp {} models/clews/checkpoint.pt

    !if [ ! -f models/clews/checkpoint.pt ]; then find downloaded_checkpoints -name "*.ckpt" | head -n 1 | xargs -I {} cp {} models/clews/checkpoint.pt; fi

    !rm clews_checkpoints.zip
    !rm -rf downloaded_checkpoints
else:
    print("\nCLEWS model already exists. Skipping download.")

print("\nSaved Models Successfully!")

/content
WEALY model already exists. Skipping download.

CLEWS model already exists. Skipping download.

Saved Models Successfully!


In [ ]:
print("Starting extraction for CLEWS...")
!PYTHONPATH=. python src/inference/extract_clews.py
print("Success!")

Starting extraction for CLEWS...
Output parquet: /content/drive/MyDrive/Plagiarism-Detection-System/data/clews_mgeldm_embeddings.parquet
data/generated_audio/mgeldm: 3033 wav files
data/dsp_variants/mgeldm: 30321 wav files
TOTAL wav files discovered: 33354
Loading existing parquet file: data/clews_mgeldm_embeddings.parquet
Found 25865 already processed files.
Existing parquet rows on disk: 25865
Remaining files to process: 7490
[Checkpoint:CLEWS] Saved 26065 rows -> /content/drive/MyDrive/Plagiarism-Detection-System/data/clews_mgeldm_embeddings.parquet
[Checkpoint:CLEWS] Saved 26265 rows -> /content/drive/MyDrive/Plagiarism-Detection-System/data/clews_mgeldm_embeddings.parquet
[Checkpoint:CLEWS] Saved 26465 rows -> /content/drive/MyDrive/Plagiarism-Detection-System/data/clews_mgeldm_embeddings.parquet
[Checkpoint:CLEWS] Saved 26665 rows -> /content/drive/MyDrive/Plagiarism-Detection-System/data/clews_mgeldm_embeddings.parquet
[Checkpoint:CLEWS] Saved 26865 rows -> /content/drive/MyDriv

In [ ]:
print("Starting extraction for WEALY...")
!PYTHONPATH=. python src/inference/extract_wealy.py
print("Success!")

Starting extraction for WEALY...
Output parquet: /content/drive/MyDrive/Plagiarism-Detection-System/data/wealy_mgeldm_embeddings.parquet
Loading Whisper model on cuda...
100%|█████████████████████████████████████| 1.51G/1.51G [00:43<00:00, 37.4MiB/s]
Initializing WEALY model...
/content/drive/MyDrive/Plagiarism-Detection-System/src/utils/wealy_lib.py:75: UserWarning: enable_nested_tensor is True, but self.use_nested_tensor is False because encoder_layer.norm_first was True
  self.transformer = nn.TransformerEncoder(encoder_layer, num_layers=conf.num_transformer_blocks)
	Missing key(s) in state_dict: "positional_encoding.pe". 
Filling missing keys from model init: ['positional_encoding.pe']
data/generated_audio/mgeldm: 3033 wav files
data/dsp_variants/mgeldm: 30321 wav files
TOTAL wav files discovered: 33354
Loading existing parquet file: data/wealy_mgeldm_embeddings.parquet
Found 66090 already processed files.
Existing parquet rows on disk: 66090
Remaining files to process: 7491
[Check

In [ ]:
print("Starting vocal detection...")
!PYTHONPATH=. python src/inference/vocal_detection.py
print("Success!")

Starting vocal detection...
[INFO] Scanning Demucs stems under: data/separated_segment_smp/mdx_extra_q
[INFO] Found 1219 vocals.wav stems total.
[INFO] Remaining: 1219 stems to process.

Vocal detection:  16% 199/1219 [01:46<09:25,  1.80it/s][Checkpoint] Saved 200 rows.
Vocal detection:  33% 399/1219 [03:22<10:22,  1.32it/s][Checkpoint] Saved 400 rows.
Vocal detection:  49% 599/1219 [04:56<03:45,  2.74it/s][Checkpoint] Saved 600 rows.
Vocal detection:  66% 799/1219 [06:41<03:31,  1.99it/s][Checkpoint] Saved 800 rows.
Vocal detection:  82% 999/1219 [08:39<02:11,  1.67it/s][Checkpoint] Saved 1000 rows.
Vocal detection:  98% 1199/1219 [10:13<00:09,  2.10it/s][Checkpoint] Saved 1200 rows.
Vocal detection: 100% 1219/1219 [10:23<00:00,  1.96it/s]

[INFO] Done. Results saved to: data/vocal_ratios_source.csv

Summary:
  Total sources       : 1219
  vocal_valid=True    : 1167 (95.7%)
  vocal_valid=False   : 52 (4.3%)

Thresholds used:
  VOCAL_RATIO_THRESHOLD  : 0.3
  ACTIVE_RATIO_THRESHOLD : 0.

In [ ]:
print("Starting pair building...")
!PYTHONPATH=. python src/evaluation/build_pairs.py
print("Success!")

Starting pair building...
Traceback (most recent call last):
  File "/content/drive/MyDrive/Plagiarism-Detection-System/src/evaluation/build_pairs.py", line 22, in <module>
    import torch
  File "/usr/local/lib/python3.12/dist-packages/torch/__init__.py", line 430, in <module>
    _load_global_deps()
  File "/usr/local/lib/python3.12/dist-packages/torch/__init__.py", line 381, in _load_global_deps
    _preload_cuda_deps()
  File "/usr/local/lib/python3.12/dist-packages/torch/__init__.py", line 348, in _preload_cuda_deps
    _preload_cuda_lib(lib_folder, lib_name)
  File "/usr/local/lib/python3.12/dist-packages/torch/__init__.py", line 318, in _preload_cuda_lib
    ctypes.CDLL(lib_path)
  File "/usr/lib/python3.12/ctypes/__init__.py", line 379, in __init__
    self._handle = _dlopen(self._name, mode)
                   ^^^^^^^^^^^^^^^^^^^^^^^^^
KeyboardInterrupt
^C
Success!
